# KSSL

> Various Transforms to be piped to create a DataLoader

In [ ]:
#| default_exp data.transforms.kssl

In [ ]:
#| hide
from nbdev.showdoc import *
from nbdev.cli import *

In [ ]:
#| export
from fastai.vision.all import *
from fastai.data.all import *
from pathlib import Path
import pandas as pd

from __future__ import annotations

## Input (spectra)

In [ ]:
#| export
def get_spectra_files(path:Path, # Path to the a kssl folder (contains spectra replicates and wet chemistry)
                     ) -> list: # List of spectra replicates (if any)
    "Return paths of spectra `.csv` files in path"
    return [f for f in path.ls() if re.match('\d', f.name)]

In [ ]:
#|eval: false
path = Path('../_data/kssl-mirs')

# Selecting only Kex > 0 values
fnames = L(filter(lambda x: get_target(x, analytes=[725]).item() > 0, fnames)); fnames

(#42443) [Path('../_data/kssl-mirs/180338'),Path('../_data/kssl-mirs/172221'),Path('../_data/kssl-mirs/177753'),Path('../_data/kssl-mirs/184798'),Path('../_data/kssl-mirs/53759'),Path('../_data/kssl-mirs/74947'),Path('../_data/kssl-mirs/176681'),Path('../_data/kssl-mirs/1855'),Path('../_data/kssl-mirs/175004'),Path('../_data/kssl-mirs/34499')...]

In [ ]:
#|eval: false
get_spectra_files(fnames[1])

[Path('../_data/kssl-mirs/172221/240596XS01.csv'),
 Path('../_data/kssl-mirs/172221/240596XS03.csv'),
 Path('../_data/kssl-mirs/172221/240596XS02.csv'),
 Path('../_data/kssl-mirs/172221/240596XS04.csv')]

In [ ]:
#| export
def to_spectra(fnames:list, # List of paths to individual spectrum file
              ) -> torch.Tensor: # Spectra
    "Transform list of paths to spectra to list of torch arrays"
    n = pd.read_csv(fnames[0]).shape[0]
    m = len(fnames)
    x = np.empty((m,n))
    for i, fname in enumerate(fnames):
        x[i,:] = pd.read_csv(fname)['absorbance'].values
    return tensor(x)

In [ ]:
#|eval: false
to_spectra([Path('../_data/kssl-mirs/172221/240596XS01.csv')])

tensor([[0.0685, 0.0689, 0.0693,  ..., 1.4879, 1.4850, 1.4844]])

In [ ]:
#| export
def rand_w_avg(x: list, # List of spectra as tensors
              ) -> torch.tensor: # Weighted random averaged spectrum
    "Transform spectra replicates taking their random weighted averages"
    n = len(x)
    def weights(n):
        weights = torch.rand(n)
        return (weights/weights.sum()).unsqueeze(dim=0)
    return torch.matmul(weights(n), x)

## Target (Soil properties)

In [ ]:
#| export
def get_target(path:Path, # Path to the a kssl folder (contains spectra replicates and wet chemistry)
               analytes:list|None=None, # List of analytes (soil properties) of interest
              ) -> torch.tensor: # Analytes as tensor
    path_target = [f for f in path.ls() if re.match('target', f.name)][0]
    df = pd.read_csv(path_target)
    if analytes:
        df = df[df.analyte.isin(analytes)]
    return tensor(df['value'].values)

In [ ]:
#|eval: false
get_target(fnames[0])

tensor([ 2.8077, 22.0330, 13.7791])

## How to use these transforms?

1. First create two // pipes (one for the features and one for the targets):

In [ ]:
#|eval: false
x_tfms = [get_spectra_files, to_spectra, rand_w_avg]
y_tfms = [partial(get_target, analytes=[725]), torch.log10]

2. Create your splits and create a Fastai `Datasets`:

In [ ]:
#|eval: false
splits = RandomSplitter(seed=42)(fnames)
dsets = Datasets(fnames, [x_tfms, y_tfms], splits=splits)

3. Then you get your Dataloader:

In [ ]:
#|eval: false
dls = dsets.dataloaders(bs=16)

In [ ]:
#|eval: false
dls.train.one_batch()[0].shape

torch.Size([16, 4, 1700])

In [ ]:
#|eval: false
dls.train.one_batch()[1]

tensor([[-0.3121],
        [-0.6380],
        [-0.8497],
        [ 0.1269],
        [ 0.1424],
        [-0.5875],
        [-0.1780],
        [-0.2884],
        [-0.2666],
        [-0.4834],
        [-0.7389],
        [-1.4153],
        [-0.4204],
        [-0.5743],
        [-0.2155],
        [-0.1279]])